# 🎓 Student Performance Prediction

Predicting students' final grade (G3) using the UCI Student Performance Dataset.

**Dataset:** [Kaggle - Student Performance Data](https://www.kaggle.com/datasets/devansodariya/student-performance-data)

**Target:** G3 (Final Grade, 0–20)

**Features dropped:**
- `G1`, `G2` — intermediate period grades. These are direct numerical proxies of G3 and would cause **data leakage** (the model would learn to just copy them, not learn real patterns).
- `school` — identifier of which school the student attends (GP or MS). Not a behavioural/demographic predictor and adds no generalisable signal.

**Approach:**
- EDA + Feature Engineering
- Multiple Algorithm Comparison
- Best Model Selection (Random Forest)
- Overfitting Detection
- Test Cases & Evaluation

## 1. Install & Import Libraries

In [ ]:
# !pip install scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import unittest

print('✅ All libraries imported successfully')

## 2. Load & Clean Dataset

In [ ]:
import os

def load_student_data(path='student_data.csv'):
    """Load the student dataset, auto-detecting the separator."""
    if not os.path.exists(path):
        url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/student-mat.csv'
        print(f'File not found locally, trying URL: {url}')
        raw = pd.read_csv(url, sep=None, engine='python')
    else:
        with open(path, 'r') as f:
            first_line = f.readline()
        sep = ';' if ';' in first_line else ','
        print(f'Detected separator: {repr(sep)}')
        raw = pd.read_csv(path, sep=sep)

    if raw.shape[1] == 1:
        raise ValueError(
            f'Dataset loaded with only 1 column — separator was wrong.\n'
            f'Fix: pd.read_csv(path, sep=";")'
        )
    return raw


df_raw = load_student_data('student_data.csv')
print(f'✅ Dataset loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')

# ── Drop leaky and irrelevant features ──────────────────────────────────
# G1 & G2 : intermediate period grades — direct numerical proxies of G3 (data leakage)
# school   : school identifier (GP / MS) — not a generalisable predictor
COLS_TO_DROP = ['G1', 'G2', 'school']
df = df_raw.drop(columns=COLS_TO_DROP)

print(f'Features after dropping {COLS_TO_DROP}: {df.shape[1] - 1} features + 1 target')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Shape:', df.shape)
print('\nData Types:\n', df.dtypes)
print('\nMissing Values:\n', df.isnull().sum())
print('\nTarget (G3) Stats:\n', df['G3'].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# G3 distribution
axes[0, 0].hist(df['G3'], bins=21, color='steelblue', edgecolor='white')
axes[0, 0].set_title('G3 (Final Grade) Distribution')
axes[0, 0].set_xlabel('Grade')
axes[0, 0].set_ylabel('Count')

# Absences vs G3
axes[0, 1].scatter(df['absences'], df['G3'], alpha=0.4, color='steelblue')
axes[0, 1].set_title('Absences vs G3')
axes[0, 1].set_xlabel('Absences')
axes[0, 1].set_ylabel('G3')

# Study time vs G3
axes[1, 0].boxplot(
    [df[df['studytime'] == st]['G3'].values for st in sorted(df['studytime'].unique())],
    labels=sorted(df['studytime'].unique())
)
axes[1, 0].set_title('Study Time vs G3')
axes[1, 0].set_xlabel('Study Time (1=<2h … 4=>10h)')
axes[1, 0].set_ylabel('G3')

# Failures vs G3
axes[1, 1].boxplot(
    [df[df['failures'] == f]['G3'].values for f in sorted(df['failures'].unique())],
    labels=sorted(df['failures'].unique())
)
axes[1, 1].set_title('Past Failures vs G3')
axes[1, 1].set_xlabel('Number of Past Failures')
axes[1, 1].set_ylabel('G3')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150)
plt.show()
print('EDA plots saved.')

## 4. Correlation Heatmap (Numeric Features Only)

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap (numeric features, G1/G2 removed)')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()
print('Heatmap saved.')

## 5. Feature Engineering & Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

df_model = df.copy()

# Encode all remaining categorical (object) columns
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include=['object']).columns.tolist()
print('Encoding categorical columns:', cat_cols)
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Remove rows where the student did not sit the final exam (G3 == 0)
df_model = df_model[df_model['G3'] > 0]
print(f'Rows after removing G3==0: {len(df_model)}')

X = df_model.drop(columns=['G3'])
y = df_model['G3']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape}, Test: {X_test_sc.shape}')

## 6. Model Training & Comparison

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Ridge'             : Ridge(),
    'Lasso'             : Lasso(),
    'Decision Tree'     : DecisionTreeRegressor(random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR'               : SVR(),
    'KNN'               : KNeighborsRegressor(),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f'{name:25s} | RMSE={rmse:.3f}  MAE={mae:.3f}  R²={r2:.3f}')

best_name = min(results, key=lambda k: results[k]['RMSE'])
print(f'\n🏆 Best model by RMSE: {best_name}')